# Simple-KNN in 3D Gaussian Splatting

3DGS의 커스텀 CUDA 확장 중 하나인 **simple-knn**에 대한 설명입니다.

## 개요

**simple-knn**은 3D 포인트 클라우드에서 **K=3 최근접 이웃까지의 평균 거리**를 계산하는 커스텀 CUDA 확장입니다.

### 3DGS에서의 실제 사용 위치

> ⚠️ **중요**: `distCUDA2`는 **초기화 단계에서 단 한 번만** 사용됩니다.  
> Adaptive Densification (split/clone)에는 사용되지 **않습니다**.

#### Adaptive Densification에서 KNN이 필요 없는 이유

Adaptive Densification (split/clone)은 **KNN 거리가 아닌 다른 기준**으로 동작합니다:

| 연산 | 조건 | Scale 처리 |
|------|------|------------|
| **Split** | gradient 높음 + scale **큼** | 부모 scale ÷ 1.6 |
| **Clone** | gradient 높음 + scale **작음** | 부모 scale 그대로 복사 |

In [ ]:
# scene/gaussian_model.py - densify_and_split() / densify_and_clone()

# Split/Clone 조건
grads = self.xyz_gradient_accum / self.denom
selected_pts_mask = grads >= grad_threshold  # gradient 기반

# Split: 큰 Gaussian (scale > threshold)
selected_pts_mask = torch.logical_and(selected_pts_mask,
    torch.max(self.get_scaling, dim=1).values > percent_dense * scene_extent)

# Clone: 작은 Gaussian (scale < threshold)  
selected_pts_mask = torch.logical_and(selected_pts_mask,
    torch.max(self.get_scaling, dim=1).values <= percent_dense * scene_extent)



**KNN이 필요 없는 이유**:

1. **초기화 시 이미 설정됨**: KNN으로 계산한 초기 scale이 학습의 시작점
2. **Scale은 learnable parameter**: 학습 중 gradient descent로 자연스럽게 조정됨
3. **Split/Clone은 현재 scale 기반**: 이웃 거리가 아닌 현재 Gaussian의 scale 사용
4. **계산 비용**: 매 iteration마다 KNN 재계산은 비효율적

즉, KNN은 "**좋은 시작점**"을 제공하고, 이후 최적화는 gradient 기반으로 진행됩니다.

#### 사용 위치: `scene/gaussian_model.py`

`distCUDA2`는 3DGS 전체 코드베이스에서 **오직 한 곳**에서만 사용됩니다:

In [ ]:
# scene/gaussian_model.py - GaussianModel.create_from_pcd() 메서드 내부
# Line 159

from simple_knn._C import distCUDA2

def create_from_pcd(self, pcd : BasicPointCloud, cam_infos : CameraInfo, spatial_lr_scale : float):
    self.spatial_lr_scale = spatial_lr_scale
    fused_point_cloud = torch.tensor(np.asarray(pcd.points)).float().cuda()
    fused_color = RGB2SH(torch.tensor(np.asarray(pcd.colors)).float().cuda())
    
    # ... features 초기화 ...
    
    print("Number of points at initialisation : ", fused_point_cloud.shape[0])

    # ========== distCUDA2 사용 위치 ==========
    # K=3 최근접 이웃까지의 평균 제곱 거리 계산
    dist2 = torch.clamp_min(
        distCUDA2(torch.from_numpy(np.asarray(pcd.points)).float().cuda()), 
        0.0000001  # 0으로 나누기 방지
    )
    # 평균 거리를 log scale로 변환하여 초기 scale로 사용
    scales = torch.log(torch.sqrt(dist2))[...,None].repeat(1, 3)
    # ==========================================
    
    rots = torch.zeros((fused_point_cloud.shape[0], 4), device="cuda")
    rots[:, 0] = 1  # identity quaternion

    opacities = self.inverse_opacity_activation(
        0.1 * torch.ones((fused_point_cloud.shape[0], 1), dtype=torch.float, device="cuda")
    )

    # learnable parameters로 등록
    self._xyz = nn.Parameter(fused_point_cloud.requires_grad_(True))
    self._scaling = nn.Parameter(scales.requires_grad_(True))
    self._rotation = nn.Parameter(rots.requires_grad_(True))
    self._opacity = nn.Parameter(opacities.requires_grad_(True))
    # ...



#### 왜 이 방식을 사용하는가?

**목적**: 초기 포인트 클라우드의 각 점에서 K=3 최근접 이웃까지의 평균 제곱 거리를 계산하여, **Gaussian의 초기 scale을 설정**합니다.

**직관**:
- 밀집한 영역 → 이웃 거리 작음 → 작은 Gaussian
- 희소한 영역 → 이웃 거리 큼 → 큰 Gaussian

**수식**:

```
초기 scale = log(sqrt(mean_squared_distance_to_3_nearest_neighbors))
```



이렇게 하면 포인트 밀도에 적응적인 초기 Gaussian 크기를 설정할 수 있습니다.

---

## API

### 주요 함수: `distCUDA2`

In [ ]:
from simple_knn._C import distCUDA2

def distCUDA2(points: torch.Tensor) -> torch.Tensor:
    """
    각 포인트의 K=3 최근접 이웃까지의 평균 제곱 거리 계산
    
    Args:
        points: [N, 3] 3D 포인트들 (CUDA tensor, float32)
    
    Returns:
        mean_dist2: [N] 각 포인트의 평균 제곱 이웃 거리
    
    Note:
        - K=3 고정 (하드코딩)
        - 제곱 거리의 평균 반환 (sqrt 하지 않음)
        - 자기 자신은 제외
    """



### 사용 예제

In [ ]:
import torch
from simple_knn._C import distCUDA2

# 포인트 생성
points = torch.rand(10000, 3).cuda()

# K=3 최근접 이웃까지의 평균 제곱 거리
avg_squared_distances = distCUDA2(points)

print(f"Shape: {avg_squared_distances.shape}")  # [10000]
print(f"Mean distance: {torch.sqrt(avg_squared_distances.mean()):.4f}")



---

## CUDA 구현 분석

### 파일 구조

```
submodules/simple-knn/
├── setup.py              # Build configuration
├── simple_knn.h          # C++ header
├── simple_knn.cu         # 핵심 CUDA 구현 (Morton code, KNN)
├── spatial.h             # PyTorch binding header
├── spatial.cu            # PyTorch binding 구현
├── ext.cpp               # PYBIND11 module definition
└── simple_knn/
    └── __init__.py       # Python package
```




### KNN (K-Nearest Neighbors) 알고리듬 배경

#### 일반적인 KNN 알고리듬

KNN(K-Nearest Neighbors)은 주어진 쿼리 포인트에서 가장 가까운 K개의 이웃을 찾는 알고리듬입니다.

**Naive 접근법**: O(N²)

```
for each query point q:
    for each point p in dataset:
        calculate distance(q, p)
    sort distances and pick K smallest
```



이 방법은 N개 포인트에 대해 모든 쌍의 거리를 계산하므로 O(N²) 시간 복잡도를 가집니다.

**가속 구조 기반 접근법**: O(N log N)
- **KD-Tree**: 공간을 재귀적으로 분할하여 탐색 범위 제한
- **Octree**: 3D 공간을 8개 자식으로 분할
- **Ball Tree**: 중첩 가능한 구로 공간 분할

#### simple-knn의 접근법: Morton Code + Box Culling

simple-knn은 GPU 병렬 처리에 최적화된 방식을 사용합니다:

1. **Morton Code 정렬**: 공간 지역성을 유지하면서 1D로 정렬
2. **Box Culling**: 1024개씩 그룹화된 박스의 min/max를 미리 계산하여 불필요한 거리 계산 skip

이 방식은 KD-Tree보다 구현이 간단하고, GPU의 SIMT 구조에 더 적합합니다.

---

### 알고리듬 상세

simple-knn은 **Morton code 기반 공간 정렬 + Box culling**을 사용합니다:

1. **Bounding Box 계산**: 전체 포인트의 min/max 좌표 (CUB Reduce)
2. **Morton Code 계산**: 3D 좌표를 1D 정수로 변환 (공간 지역성 유지)
3. **Radix Sort**: Morton code로 포인트 정렬 (CUB RadixSort)
4. **Box 분할**: BOX_SIZE(1024)개씩 그룹화하여 각 box의 min/max 계산
5. **KNN 탐색**: Box culling으로 후보 필터링 후 K=3 최근접 이웃 탐색

### Morton Code (Z-order Curve)

In [ ]:
// simple_knn.cu (Lines 44-54)
__host__ __device__ uint32_t prepMorton(uint32_t x)
{
    x = (x | (x << 16)) & 0x030000FF;
    x = (x | (x << 8)) & 0x0300F00F;
    x = (x | (x << 4)) & 0x030C30C3;
    x = (x | (x << 2)) & 0x09249249;
    return x;
}

__host__ __device__ uint32_t coord2Morton(float3 coord, float3 minn, float3 maxx)
{
    // 좌표를 0-1023 범위로 정규화 후 Morton code 생성
    uint32_t x = prepMorton(((coord.x - minn.x) / (maxx.x - minn.x)) * ((1 << 10) - 1));
    uint32_t y = prepMorton(((coord.y - minn.y) / (maxx.y - minn.y)) * ((1 << 10) - 1));
    uint32_t z = prepMorton(((coord.z - minn.z) / (maxx.z - minn.z)) * ((1 << 10) - 1));
    return x | (y << 1) | (z << 2);  // x, y, z 비트 인터리빙
}



**Morton Code의 장점**:
- 공간적으로 가까운 점들이 1D 순서에서도 가까이 위치
- 정렬 후 인접 포인트 탐색이 효율적

### K=3 최근접 이웃 업데이트

In [ ]:
// simple_knn.cu (Lines 132-147)
template<int K>
__device__ void updateKBest(const float3& ref, const float3& point, float* knn)
{
    float3 d = { point.x - ref.x, point.y - ref.y, point.z - ref.z };
    float dist = d.x * d.x + d.y * d.y + d.z * d.z;
    
    // Insertion sort로 K개 최소 거리 유지
    for (int j = 0; j < K; j++)
    {
        if (knn[j] > dist)
        {
            float t = knn[j];
            knn[j] = dist;
            dist = t;
        }
    }
}



### Box Culling을 사용한 KNN 탐색

In [ ]:
// simple_knn.cu (Lines 149-180)
__global__ void boxMeanDist(uint32_t P, float3* points, uint32_t* indices, 
                            MinMax* boxes, float* dists)
{
    int idx = cg::this_grid().thread_rank();
    if (idx >= P) return;

    float3 point = points[indices[idx]];
    float best[3] = { FLT_MAX, FLT_MAX, FLT_MAX };

    // Step 1: 인접 포인트로 초기 reject threshold 설정
    for (int i = max(0, idx - 3); i <= min(P - 1, idx + 3); i++)
    {
        if (i == idx) continue;
        updateKBest<3>(point, points[indices[i]], best);
    }
    float reject = best[2];  // 3번째로 가까운 거리 = rejection threshold
    
    // Step 2: 모든 box 순회하며 KNN 탐색
    best[0] = best[1] = best[2] = FLT_MAX;  // reset
    
    for (int b = 0; b < (P + BOX_SIZE - 1) / BOX_SIZE; b++)
    {
        MinMax box = boxes[b];
        float dist = distBoxPoint(box, point);  // point와 box 간 최소 거리
        
        // Box culling: box가 이미 찾은 K번째보다 멀면 skip
        if (dist > reject || dist > best[2])
            continue;

        // Box 내 모든 포인트 검사
        for (int i = b * BOX_SIZE; i < min(P, (b + 1) * BOX_SIZE); i++)
        {
            if (i == idx) continue;
            updateKBest<3>(point, points[indices[i]], best);
        }
    }
    
    // 평균 계산
    dists[indices[idx]] = (best[0] + best[1] + best[2]) / 3.0f;
}



### 전체 파이프라인

In [ ]:
// simple_knn.cu (Lines 182-222)
void SimpleKNN::knn(int P, float3* points, float* meanDists)
{
    // 1. Min/Max 계산 (CUB Reduce)
    cub::DeviceReduce::Reduce(..., CustomMin(), ...);  // minn
    cub::DeviceReduce::Reduce(..., CustomMax(), ...);  // maxx

    // 2. Morton Code 계산
    coord2Morton<<<(P + 255) / 256, 256>>>(P, points, minn, maxx, morton.data());

    // 3. Radix Sort (Morton code로 포인트 정렬)
    cub::DeviceRadixSort::SortPairs(..., morton, indices, ...);

    // 4. Box별 min/max 계산
    uint32_t num_boxes = (P + BOX_SIZE - 1) / BOX_SIZE;  // BOX_SIZE = 1024
    boxMinMax<<<num_boxes, BOX_SIZE>>>(P, points, indices_sorted, boxes);

    // 5. K=3 최근접 이웃 탐색
    boxMeanDist<<<num_boxes, BOX_SIZE>>>(P, points, indices_sorted, boxes, meanDists);
}



---

## PyTorch Binding

### spatial.cu

In [ ]:
// spatial.cu (전체 코드)
#include "spatial.h"
#include "simple_knn.h"

torch::Tensor distCUDA2(const torch::Tensor& points)
{
    const int P = points.size(0);
    
    auto float_opts = points.options().dtype(torch::kFloat32);
    torch::Tensor means = torch::full({P}, 0.0, float_opts);
    
    SimpleKNN::knn(P, (float3*)points.contiguous().data<float>(), 
                   means.contiguous().data<float>());
    
    return means;
}



### ext.cpp

In [ ]:
// ext.cpp (전체 코드)
#include <torch/extension.h>
#include "spatial.h"

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("distCUDA2", &distCUDA2);
}



---

## 설치

### 요구사항

- CUDA Toolkit 11.8 이상
- PyTorch with CUDA support
- Visual Studio (Windows) / GCC (Linux)

### 설치 방법

In [ ]:
cd gaussian-splatting/submodules/simple-knn
pip install . --no-build-isolation



### 테스트

In [ ]:
import torch
from simple_knn._C import distCUDA2

points = torch.rand(1000, 3).cuda()
dist2 = distCUDA2(points)
print(f"Success! Output shape: {dist2.shape}")



---

## 요약

### simple-knn의 실제 역할

| 항목 | 설명 |
|------|------|
| **사용 시점** | 초기화 단계에서 **한 번** |
| **목적** | Gaussian 초기 scale 설정 |
| **입력** | 초기 포인트 클라우드 [N, 3] |
| **출력** | 평균 제곱 이웃 거리 [N] |
| **K 값** | 3 (하드코딩) |

### 핵심 기술

- **Morton Code**: 공간 지역성을 유지하는 1D 정렬
- **Box Culling**: 불필요한 거리 계산 skip
- **CUB Library**: GPU primitives (Reduce, RadixSort)

### ⚠️ 주의사항

`distCUDA2`는 **Adaptive Densification (split/clone)에는 사용되지 않습니다**.

Split/clone 결정은 다음을 기반으로 합니다:
- `xyz_gradient_accum`: 위치 gradient 누적값
- `percent_dense * scene_extent`: Gaussian 크기 threshold
- Gradient threshold 비교

---

## 참고자료

### 코드 위치

- **CUDA 구현**: [simple_knn.cu](../../references/gaussian-splatting/submodules/simple-knn/simple_knn.cu)
- **PyTorch 바인딩**: [spatial.cu](../../references/gaussian-splatting/submodules/simple-knn/spatial.cu)
- **사용 위치**: [gaussian_model.py Line 159](../../references/gaussian-splatting/scene/gaussian_model.py#L159)

### 관련 자료

1. **3D Gaussian Splatting 논문**: [repo-sam.inria.fr/fungraph/3d-gaussian-splatting](https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/)
2. **Morton Code / Z-order Curve**: [Wikipedia](https://en.wikipedia.org/wiki/Z-order_curve)
3. **CUB Library**: [NVIDIA CUB](https://nvlabs.github.io/cub/)